<a href="https://colab.research.google.com/github/yosie111/Shulchan_Aruch_RAG_1/blob/main/SA_RAG_Stage4_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shulchan Aruch RAG — Stage 4: Generation + End-to-End Evaluation

## ארכיטקטורה (מבוססת על ממצאי Stage 2+3):
```
שאלה → e5-large hybrid (top-25) → bge-reranker (top-5) → LLM → תשובה
```

## למה הקונפיגורציה הזו:
- e5-large hybrid: ceiling=95.1% ב-K=25 (הגבוה ביותר)
- bge-reranker-v2-m3: שיפור של +8% ב-Recall@5 ב-Stage 3
- mxbai-reranker: נפסל (הזיק)

## קלט:
- `shulchan_aruch_v2.jsonl`
- `על_שוע_חלק_א.xlsx`
- **API key** (Anthropic / OpenAI) או מודל מקומי

## הוראות:
1. Runtime → GPU
2. הכנס API key ב-Secrets (או שנה ל-local)
3. הרץ תאים 1-7


In [1]:
# ============================================================
# תא 1 — התקנה
# ============================================================
!pip install -q sentence-transformers chromadb openpyxl rank-bm25 anthropic openai

import json, re, os, time
import numpy as np
from pathlib import Path
from collections import defaultdict

import openpyxl, chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

try:
    from google.colab import files as colab_files, userdata
    IN_COLAB = True
except Exception:
    IN_COLAB = False

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda': print(f'  GPU: {torch.cuda.get_device_name(0)}')
print('✅ תא 1')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 146.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 164.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# ============================================================
# תא 2 — טעינת נתונים + retrieval pipeline מלא
#   e5-large hybrid (top-25) → bge-reranker (top-5)
# ============================================================

# --- A. JSONL ---
JSONL_FILE = None
for p in sorted(Path('.').glob('*.jsonl')):
    JSONL_FILE = str(p); break
if JSONL_FILE is None and IN_COLAB:
    print('העלה JSONL:'); uploaded = colab_files.upload()
    for n in uploaded:
        if n.endswith('.jsonl'): JSONL_FILE = n; break
if not JSONL_FILE: raise FileNotFoundError('JSONL not found')

chunks = []
with open(JSONL_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): chunks.append(json.loads(line))
content_chunks = [c for c in chunks if c.get('metadata',{}).get('doc_type') != 'hilchot_index']
print(f'📄 {len(content_chunks)} chunks')

# --- B. Excel ---
EVAL_FILE = None
for p in sorted(Path('.').glob('*.xlsx')):
    EVAL_FILE = str(p); break
if EVAL_FILE is None and IN_COLAB:
    print('העלה Excel:'); uploaded = colab_files.upload()
    for n in uploaded:
        if n.endswith('.xlsx'): EVAL_FILE = n; break

GEMATRIA = {'א':1,'ב':2,'ג':3,'ד':4,'ה':5,'ו':6,'ז':7,'ח':8,'ט':9,
    'י':10,'כ':20,'ל':30,'מ':40,'נ':50,'ס':60,'ע':70,'פ':80,'צ':90,
    'ק':100,'ר':200,'ש':300,'ת':400,'ך':20,'ם':40,'ן':50,'ף':80,'ץ':90}
def heb2num(s):
    return sum(GEMATRIA.get(c,0) for c in re.sub(r'["\'\'  ׳״]','',s or ''))

eval_questions = []
if EVAL_FILE:
    wb = openpyxl.load_workbook(EVAL_FILE); ws = wb.active
    for row in range(2, ws.max_row + 1):
        q = ws.cell(row,1).value
        if not q: continue
        src = ws.cell(row,3).value or ''
        sm = re.search(r'סימן\s+([א-ת]+)', src)
        sem = re.search(r'סעיף\s+([א-ת]+)', src)
        eval_questions.append({
            'question': q, 'answer': ws.cell(row,2).value or '',
            'source': src, 'document': ws.cell(row,4).value or '',
            'gold_siman': heb2num(sm.group(1)) if sm else 0,
            'gold_seif': heb2num(sem.group(1)) if sem else 0,
        })
    print(f'📝 {len(eval_questions)} שאלות')

# --- C. Gold mapping ---
def find_gold_chunk(q, chs):
    gs, gse = q['gold_siman'], q['gold_seif']
    for c in chs:
        h = c.get('metadata',{}).get('hierarchy',{})
        if h.get('level_1_num') == gs:
            if gse in h.get('level_2_nums',[]) or gse == 0:
                return c['doc_id']
    for c in chs:
        if c.get('metadata',{}).get('hierarchy',{}).get('level_1_num') == gs:
            return c['doc_id']
    return None

for q in eval_questions:
    q['gold_chunk_id'] = find_gold_chunk(q, content_chunks)

# --- D. Shared data ---
chunk_ids = [c['doc_id'] for c in content_chunks]
chunk_texts_emb = [c['content']['text_for_embedding'] for c in content_chunks]
chunk_texts_bm25 = [c['content']['text_for_bm25'] for c in content_chunks]
chunk_texts_llm = [c['content']['text_for_llm'] for c in content_chunks]
id_to_llm = {c['doc_id']: c['content']['text_for_llm'] for c in content_chunks}

# --- E. e5-large embedding ---
E5_DOC_PREFIX = 'passage: '
E5_QUERY_PREFIX = 'query: '
print('Loading e5-large...')
embed_model = SentenceTransformer('intfloat/multilingual-e5-large', device=DEVICE)
print('Encoding chunks...')
doc_vectors = embed_model.encode(
    [E5_DOC_PREFIX + t for t in chunk_texts_emb],
    show_progress_bar=True, normalize_embeddings=True
)

# ChromaDB
chroma = chromadb.Client()
try: chroma.delete_collection('sa_gen')
except: pass
collection = chroma.create_collection('sa_gen', metadata={'hnsw:space': 'cosine'})
collection.add(
    ids=chunk_ids,
    embeddings=[v.tolist() for v in doc_vectors],
    documents=chunk_texts_llm,
)

# --- F. BM25 ---
print('Building BM25...')
tokenized_corpus = [t.split() for t in chunk_texts_bm25]
bm25_index = BM25Okapi(tokenized_corpus)

# --- G. Reranker ---
print('Loading reranker...')
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=DEVICE)

# --- H. Retrieve function ---
RRF_K = 60
TOP_N_RETRIEVE = 25
TOP_K_RERANK = 5

def retrieve(question, n_results=5):
    """Full pipeline: e5-large hybrid (top-25) → reranker (top-k)"""
    # Vector
    qvec = embed_model.encode([E5_QUERY_PREFIX + question], normalize_embeddings=True)
    vec_res = collection.query(query_embeddings=[qvec[0].tolist()], n_results=TOP_N_RETRIEVE)
    vec_ids = vec_res['ids'][0]

    # BM25
    bm25_scores = bm25_index.get_scores(question.split())
    bm25_top = np.argsort(bm25_scores)[::-1][:TOP_N_RETRIEVE]
    bm25_ids = [chunk_ids[i] for i in bm25_top]

    # RRF fusion
    rrf = defaultdict(float)
    for rank, cid in enumerate(vec_ids):
        rrf[cid] += 0.7 / (RRF_K + rank + 1)
    for rank, cid in enumerate(bm25_ids):
        rrf[cid] += 0.3 / (RRF_K + rank + 1)
    hybrid_ids = sorted(rrf, key=rrf.get, reverse=True)[:TOP_N_RETRIEVE]

    # Rerank
    candidates = [(cid, id_to_llm.get(cid, '')) for cid in hybrid_ids]
    pairs = [(question, text) for _, text in candidates]
    scores = reranker.predict(pairs)
    scored = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    top_k = scored[:n_results]

    return [{'chunk_id': cid, 'text': text, 'score': float(sc)}
            for (cid, text), sc in top_k]

# Test
test = retrieve('מתי מברכים על ציצית')
print(f'\nTest: {test[0]["chunk_id"]}')
print(f'✅ תא 2 — retriever: e5-large hybrid + reranker')


העלה JSONL:


Saving shulchan_aruch_v2 (11).jsonl to shulchan_aruch_v2 (11).jsonl
📄 1355 chunks
העלה Excel:


Saving על שוע    חלק א.xlsx to על שוע    חלק א.xlsx
📝 102 שאלות
Loading e5-large...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]


Test: sa_oc_018_1-3
✅ תא 2 — retriever: e5-large hybrid + reranker


In [4]:
# ============================================================
# תא 3 — הגדרת LLM
# ============================================================
# שנה כאן:
LLM_PROVIDER = 'local'  # 'local' / 'claude' / 'openai'

if LLM_PROVIDER == 'claude':
    import anthropic
    try:
        api_key = userdata.get('ANTHROPIC_API_KEY') if IN_COLAB else os.environ.get('ANTHROPIC_API_KEY')
    except: api_key = os.environ.get('ANTHROPIC_API_KEY')
    if not api_key: api_key = input('Anthropic API Key: ')
    llm_client = anthropic.Anthropic(api_key=api_key)
    LLM_MODEL = 'claude-sonnet-4-20250514'
    print(f'LLM: Claude ({LLM_MODEL})')

elif LLM_PROVIDER == 'openai':
    import openai
    try:
        api_key = userdata.get('OPENAI_API_KEY') if IN_COLAB else os.environ.get('OPENAI_API_KEY')
    except: api_key = os.environ.get('OPENAI_API_KEY')
    if not api_key: api_key = input('OpenAI API Key: ')
    llm_client = openai.OpenAI(api_key=api_key)
    LLM_MODEL = 'gpt-4o'
    print(f'LLM: OpenAI ({LLM_MODEL})')

elif LLM_PROVIDER == 'local':
    from transformers import pipeline as hf_pipeline
    LLM_MODEL = 'dicta-il/dictalm2.0-instruct'
    print(f'Loading {LLM_MODEL}...')
    llm_client = hf_pipeline('text-generation', model=LLM_MODEL,
                             torch_dtype=torch.bfloat16, device_map='auto',
                             max_new_tokens=500, return_full_text=False)
    print(f'LLM: Local ({LLM_MODEL})')


# === System Prompt ===
SYSTEM_PROMPT = """אתה מומחה הלכתי שעונה על שאלות אך ורק על סמך מקורות שסופקו לך.

כללים מחייבים:
1. ענה רק על סמך המקטעים שסופקו. אם התשובה לא נמצאת — אמור "לא מצאתי מקור לכך".
2. בתחילת כל תשובה ציין מקור: סימן + סעיף.
3. אם יש מחלוקת מחבר/רמ"א — ציין שתי דעות.
4. לשון הלכתית קצרה.
5. אל תוסיף מידע שאינו במקטעים.

פורמט:
מקור: [סימן X, סעיף Y]
תשובה: [תשובה קצרה]
פירוט: [הסבר מבוסס על הטקסט]"""


def build_prompt(question, retrieved_chunks):
    parts = []
    for i, ch in enumerate(retrieved_chunks, 1):
        parts.append(f'--- מקטע {i} ({ch["chunk_id"]}) ---\n{ch["text"]}')
    context = '\n\n'.join(parts)
    return f"""להלן מקטעים מתוך שולחן ערוך:\n\n{context}\n\n---\nשאלה: {question}"""


def generate_answer(question, retrieved_chunks):
    user_msg = build_prompt(question, retrieved_chunks)

    if LLM_PROVIDER == 'claude':
        resp = llm_client.messages.create(
            model=LLM_MODEL, max_tokens=500,
            system=SYSTEM_PROMPT,
            messages=[{'role': 'user', 'content': user_msg}],
        )
        return resp.content[0].text

    elif LLM_PROVIDER == 'openai':
        resp = llm_client.chat.completions.create(
            model=LLM_MODEL, max_tokens=500,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': user_msg},
            ],
        )
        return resp.choices[0].message.content

    elif LLM_PROVIDER == 'local':
        full = f'{SYSTEM_PROMPT}\n\n{user_msg}'
        result = llm_client([{'role': 'user', 'content': full}])
        return result[0]['generated_text'].strip()


# Test
print('Testing...')
test_q = 'מה צריך האדם לעשות בבוקר?'
test_ch = retrieve(test_q, n_results=3)
test_ans = generate_answer(test_q, test_ch)
print(f'Q: {test_q}')
print(f'Retrieved: {[c["chunk_id"] for c in test_ch]}')
print(f'Answer:\n{test_ans}')
print(f'\n✅ תא 3 — pipeline עובד')


Loading dicta-il/dictalm2.0-instruct...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM: Local (dicta-il/dictalm2.0-instruct)
Testing...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: מה צריך האדם לעשות בבוקר?
Retrieved: ['sa_oc_001_1-4__a', 'sa_oc_006_1__a', 'sa_oc_007_1-4']
Answer:
סעיף א מ-sa_oc_001_1-4__a: "יתגבר כארי לעמוד בבוקר לעבודת בוראו"

תשובה: האדם צריך להתגבר כארי ולקום בכל בוקר לעבודת השם יתברך.

פירוט: שולחן ערוך ממליץ על התנהגות מחזקת כדי לעורר את האדם לקום מוקדם לעבודת הבורא. זה כולל את ההשוואה לאדם המתעורר כדי לעבוד עבור מלך גדול.

סעיף ב מ-sa_oc_006_1__a: "על החורבן ועל הגלות רצויה"

תשובה: האדם צריך להתפלל על חורבן בית המקדש והגלות.

פירוט: הפסוק בשולחן ערוך ממליץ להתחנן בפני אלוהים על החורבן והגלות, אשר נחשבים לתפילות רצויות.

סעיף ד מ-sa_oc_007_1-4: "שאם יסתם הנקב מלהוציא הטפה ההיא היה קשה לו וחייב להודות"

תשובה: האדם צריך לברך "אשר יצר" על הטלת מים, אפילו לטיפה אחת, כיוון שאם הנקב היה נסתם, זה היה קשה עבורו.

פירוט: שולחן ערוך מציין שאדם חייב לברך "אשר יצר" על הטלת מים, אפילו לטיפה אחת, כיוון שזה מדגיש את הפלא שבמערכת הגוף האנושי. אם נקב כלשהו היה נסתם, הדבר היה קשה, ולכן יש להודות על כך.

✅ תא 3 — pipeline עובד


In [5]:
# ============================================================
# תא 4 — הרצת 102 שאלות end-to-end
# ============================================================

N_RETRIEVE = 5
DELAY = 0  # ללא השהיה

results = []
errors = []

print(f'Running {len(eval_questions)} questions...')
print(f'  Retriever: e5-large hybrid top-{TOP_N_RETRIEVE} → reranker top-{N_RETRIEVE}')
print(f'  LLM: {LLM_PROVIDER} ({LLM_MODEL})')
print()

t_start = time.time()
for i, q in enumerate(eval_questions):
    try:
        retrieved = retrieve(q['question'], n_results=N_RETRIEVE)
        retrieved_ids = [r['chunk_id'] for r in retrieved]
        answer = generate_answer(q['question'], retrieved)

        results.append({
            'idx': i,
            'question': q['question'],
            'gold_answer': q['answer'],
            'gold_source': q['source'],
            'gold_chunk_id': q.get('gold_chunk_id'),
            'retrieved_ids': retrieved_ids,
            'retrieval_hit': q.get('gold_chunk_id') in retrieved_ids,
            'generated_answer': answer,
        })

        hit = '✅' if q.get('gold_chunk_id') in retrieved_ids else '❌'
        if (i+1) % 10 == 0 or i < 3:
            print(f'  [{i+1:>3}/{len(eval_questions)}] {hit} {q["question"][:50]}')


    except Exception as e:
        errors.append((i, str(e)))
        print(f'  [{i+1:>3}] ❗ {str(e)[:60]}')
        time.sleep(2)

elapsed = time.time() - t_start
print(f'\n✅ Done: {len(results)} answers, {len(errors)} errors ({elapsed:.0f}s)')


Running 102 questions...
  Retriever: e5-large hybrid top-25 → reranker top-5
  LLM: local (dicta-il/dictalm2.0-instruct)



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [  1/102] ✅ מה צריך האדם לעשות בבוקר ?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [  2/102] ❌ האם מותר לאחר את זמן התפילה ?


...

  [ 90/102] ✅ כמה ברכות חייב אדם ביום ?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for op

  [100/102] ✅ האם מותר לשנות תפילין של ראש לשל יד?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ Done: 102 answers, 0 errors (470s)


In [6]:
# ============================================================
# תא 5 — מדדי הערכה
# ============================================================

def extract_cited_source(text):
    m = re.search(r'סימן\s+([א-ת\'\"״׳]+)', text)
    siman = m.group(1) if m else ''
    m2 = re.search(r'סעיף\s+([א-ת\'\"״׳]+)', text)
    seif = m2.group(1) if m2 else ''
    return siman, seif

def check_citation(result, q):
    cited_s, _ = extract_cited_source(result['generated_answer'])
    gold_src = q.get('source', '')
    if cited_s and cited_s in gold_src: return 'correct'
    elif cited_s: return 'wrong'
    else: return 'none'

def check_answer(result, q):
    gold = q.get('answer', '').strip()
    gen = result['generated_answer']
    if not gold: return 'no_gold'
    if gold in gen: return 'exact'
    gold_w = set(gold.split())
    gen_w = set(gen.split())
    overlap = len(gold_w & gen_w) / max(len(gold_w), 1)
    if overlap >= 0.5: return 'partial'
    return 'miss'

def check_halluc(result):
    ans = result['generated_answer']
    refused = any(p in ans for p in ['לא מצאתי', 'לא נמצא', 'אין במקטעים'])
    hit = result['retrieval_hit']
    if hit and not refused: return 'answered'
    if not hit and refused: return 'refused_ok'
    if not hit and not refused: return 'halluc_risk'
    return 'false_refusal'

# Compute
n = len(results)
ret_hits = sum(1 for r in results if r['retrieval_hit'])
cit = defaultdict(int)
ans = defaultdict(int)
hal = defaultdict(int)
for r in results:
    q = eval_questions[r['idx']]
    cit[check_citation(r, q)] += 1
    ans[check_answer(r, q)] += 1
    hal[check_halluc(r)] += 1

print(f'{"="*70}')
print(f'{"END-TO-END EVALUATION":^70}')
print(f'{"="*70}')
print(f'  Questions: {n}')
print(f'  Retriever: e5-large hybrid + reranker')
print(f'  LLM: {LLM_PROVIDER} ({LLM_MODEL})')
print()

print(f'--- Retrieval ---')
print(f'  Recall@{N_RETRIEVE}: {ret_hits/n:.3f} ({ret_hits}/{n})')
print()

print(f'--- Citation ---')
for k in ['correct','wrong','none']:
    print(f'  {k:<12}: {cit[k]:>3} ({100*cit[k]//n}%)')
print()

print(f'--- Answer Match ---')
for k in ['exact','partial','miss','no_gold']:
    print(f'  {k:<12}: {ans[k]:>3} ({100*ans[k]//n}%)')
print()

print(f'--- Hallucination ---')
for k in ['answered','refused_ok','halluc_risk','false_refusal']:
    print(f'  {k:<15}: {hal[k]:>3} ({100*hal[k]//n}%)')

correct_cit = cit['correct']
match = ans['exact'] + ans['partial']
safe = hal['answered'] + hal['refused_ok']
print(f'\n{"="*70}')
print(f'  📊 Citation accuracy:  {100*correct_cit//n}%')
print(f'  📊 Answer match:       {100*match//n}%')
print(f'  📊 Safety:             {100*safe//n}%')
print(f'{"="*70}')
print(f'✅ תא 5')


                        END-TO-END EVALUATION                         
  Questions: 102
  Retriever: e5-large hybrid + reranker
  LLM: local (dicta-il/dictalm2.0-instruct)

--- Retrieval ---
  Recall@5: 0.892 (91/102)

--- Citation ---
  correct     :  38 (37%)
  wrong       :  22 (21%)
  none        :  42 (41%)

--- Answer Match ---
  exact       :  35 (34%)
  partial     :  30 (29%)
  miss        :  37 (36%)
  no_gold     :   0 (0%)

--- Hallucination ---
  answered       :  88 (86%)
  refused_ok     :   0 (0%)
  halluc_risk    :  11 (10%)
  false_refusal  :   3 (2%)

  📊 Citation accuracy:  37%
  📊 Answer match:       63%
  📊 Safety:             86%
✅ תא 5


In [7]:
# ============================================================
# תא 6 — דוגמאות + שגיאות
# ============================================================

# Good examples
print(f'{"="*70}')
print(f'{"GOOD EXAMPLES":^70}')
print(f'{"="*70}')
good = [r for r in results if r['retrieval_hit'] and
        check_answer(r, eval_questions[r['idx']]) in ('exact','partial')]
for r in good[:5]:
    q = eval_questions[r['idx']]
    print(f'\nQ: {r["question"]}')
    print(f'Gold: {q["answer"]} ({q["source"]})')
    print(f'Generated:\n{r["generated_answer"][:200]}')
    print('-'*50)

# Retrieval failures
print(f'\n{"="*70}')
print(f'{"RETRIEVAL FAILURES":^70}')
print(f'{"="*70}')
ret_fail = [r for r in results if not r['retrieval_hit']]
for r in ret_fail[:5]:
    q = eval_questions[r['idx']]
    print(f'\nQ: {r["question"]}')
    print(f'Gold: {q["gold_chunk_id"]}')
    print(f'Got: {r["retrieved_ids"][:3]}')
    print(f'Answer: {r["generated_answer"][:150]}')
    print('-'*50)

# Hallucination risk
print(f'\n{"="*70}')
print(f'{"HALLUCINATION RISK":^70}')
print(f'{"="*70}')
halluc = [r for r in results if check_halluc(r) == 'halluc_risk']
if halluc:
    for r in halluc[:5]:
        q = eval_questions[r['idx']]
        print(f'\nQ: {r["question"]}')
        print(f'Gold chunk: {q["gold_chunk_id"]} (NOT retrieved)')
        print(f'Answer: {r["generated_answer"][:200]}')
        print('-'*50)
else:
    print('\n  No hallucinations detected! 🎉')

print(f'\n✅ תא 6')


                            GOOD EXAMPLES                             

Q: מה צריך האדם לעשות בבוקר ?
Gold: יתגבר כארי (סימן א, סעיף א)
Generated:
בבוקר, על האדם לעשות את הדברים הבאים:

1. להתגבר כארי לעמוד בבוקר לעבודת בוראו.
2. לכוון לשעות של חילופי המשמרות ולהקדיש תפילה על חורבן בית המקדש.
3. לברך את ברכת "אשר יצר" ולהתכוון לשבחי האדם, לריפוי
--------------------------------------------------

Q: האם מותר לומר עשרת הדברות בציבור ?
Gold: רק ביחיד (סימן א, סעיף ה)
Generated:
לא, מותר לומר עשרת הדברות בציבור (הגהת הרמ"א, סימן א, סעיף ה). לפי ההלכה, רק יחיד יכול לומר את עשרת הדברות בביתו, אבל בציבור זה נחשב לדבר שבקדושה ואסור להגיד אותם. לכן, יש להימנע מאמירת עשרת הדברות בצ
--------------------------------------------------

Q: מתי יש לומר פרשיות הקרבנות ?
Gold: רק ביום (סימן א, סעיף ו)
Generated:
המקור: [אורח חיים תרסג, סעיף א]

תשובה: בחול המועד מוציאים ספר תורה וקוראים בו ארבעה בקרבנות החג שבפרשת פנחס וביום ראשון של חולו של מועד קורא כהן וביום השני ולוי וישראל חוזר וקורא וביום השלישי

In [8]:
# ============================================================
# תא 7 — שמירה + הורדה
# ============================================================

OUTPUT_DIR = '/content/generation_output' if IN_COLAB else 'generation_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

output = {
    'config': {
        'retriever': 'e5-large hybrid top-25 + bge-reranker top-5',
        'llm': f'{LLM_PROVIDER}/{LLM_MODEL}',
        'n_questions': n,
        'n_chunks': len(content_chunks),
    },
    'metrics': {
        'retrieval_recall': ret_hits / n,
        'citation_accuracy': correct_cit / n,
        'answer_match': match / n,
        'safety': safe / n,
    },
    'results': results,
}

json_path = os.path.join(OUTPUT_DIR, 'generation_results.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print(f'📄 {json_path}')

# Summary
summary_path = os.path.join(OUTPUT_DIR, 'generation_summary.txt')
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(f'Generation Evaluation Summary\n{"="*50}\n')
    f.write(f'Retriever: e5-large hybrid + reranker\n')
    f.write(f'LLM: {LLM_PROVIDER}/{LLM_MODEL}\n')
    f.write(f'Questions: {n}\nChunks: {len(content_chunks)}\n\n')
    f.write(f'Retrieval Recall@5: {ret_hits/n:.3f}\n')
    f.write(f'Citation accuracy: {100*correct_cit//n}%\n')
    f.write(f'Answer match: {100*match//n}%\n')
    f.write(f'Safety: {100*safe//n}%\n')
print(f'📄 {summary_path}')

# List + Download
print(f'\n📂 {os.path.abspath(OUTPUT_DIR)}/')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    print(f'  📄 {fname} ({os.path.getsize(fpath):,} bytes)')

if IN_COLAB:
    print('\n⬇️  מוריד...')
    from google.colab import files as colab_files
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        colab_files.download(os.path.join(OUTPUT_DIR, fname))
    print('✅ הורדו!')
else:
    print('\n✅ סיום')


📄 /content/generation_output/generation_results.json
📄 /content/generation_output/generation_summary.txt

📂 /content/generation_output/
  📄 generation_results.json (140,476 bytes)
  📄 generation_summary.txt (267 bytes)

⬇️  מוריד...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ הורדו!


In [9]:
# ============================================================
# תא 8 — שמירת כל התוצרים + הורדה
#
# נשמרים:
#   1. vectors — embeddings של כל ה-chunks (numpy)
#   2. chunk_ids + metadata — מיפוי id → index
#   3. BM25 corpus — טוקנים לבניית BM25
#   4. generation results — תשובות + מדדים
#   5. evaluation summary
#
# בהרצה הבאה: טוענים vectors במקום לחשב מחדש
# ============================================================

import pickle

OUTPUT_DIR = '/content/rag_artifacts' if IN_COLAB else 'rag_artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === 1. Vectors (numpy) ===
vectors_path = os.path.join(OUTPUT_DIR, 'e5large_vectors.npy')
np.save(vectors_path, doc_vectors)
print(f'📄 {vectors_path} ({doc_vectors.shape}, {os.path.getsize(vectors_path):,} bytes)')

# === 2. Chunk mapping ===
chunk_map = {
    'chunk_ids': chunk_ids,
    'id_to_llm': id_to_llm,
    'metadatas': [c['metadata'] for c in content_chunks],
}
map_path = os.path.join(OUTPUT_DIR, 'chunk_map.pkl')
with open(map_path, 'wb') as f:
    pickle.dump(chunk_map, f)
print(f'📄 {map_path} ({os.path.getsize(map_path):,} bytes)')

# === 3. BM25 tokenized corpus ===
bm25_path = os.path.join(OUTPUT_DIR, 'bm25_corpus.pkl')
with open(bm25_path, 'wb') as f:
    pickle.dump(tokenized_corpus, f)
print(f'📄 {bm25_path} ({os.path.getsize(bm25_path):,} bytes)')

# === 4. Generation results (full) ===
gen_output = {
    'config': {
        'retriever': 'e5-large hybrid top-25 + bge-reranker top-5',
        'embedding_model': 'intfloat/multilingual-e5-large',
        'reranker': 'BAAI/bge-reranker-v2-m3',
        'llm': f'{LLM_PROVIDER}/{LLM_MODEL}',
        'n_questions': n,
        'n_chunks': len(content_chunks),
        'top_n_retrieve': TOP_N_RETRIEVE,
        'top_k_rerank': N_RETRIEVE,
    },
    'metrics': {
        'retrieval_recall': round(ret_hits / n, 4),
        'citation_accuracy': round(correct_cit / n, 4),
        'answer_match': round(match / n, 4),
        'safety': round(safe / n, 4),
    },
    'results': results,
}
gen_path = os.path.join(OUTPUT_DIR, 'generation_results.json')
with open(gen_path, 'w', encoding='utf-8') as f:
    json.dump(gen_output, f, ensure_ascii=False, indent=2)
print(f'📄 {gen_path} ({os.path.getsize(gen_path):,} bytes)')

# === 5. Summary ===
summary_lines = [
    'RAG Pipeline — Full Artifacts',
    '=' * 50,
    f'Embedding: intfloat/multilingual-e5-large',
    f'Vectors: {doc_vectors.shape}',
    f'Chunks: {len(content_chunks)}',
    f'Reranker: BAAI/bge-reranker-v2-m3',
    f'LLM: {LLM_PROVIDER}/{LLM_MODEL}',
    f'Questions: {n}',
    '',
    'Metrics:',
    f'  Retrieval Recall@5: {ret_hits/n:.3f}',
    f'  Citation accuracy:  {100*correct_cit//n}%',
    f'  Answer match:       {100*match//n}%',
    f'  Safety:             {100*safe//n}%',
    '',
    'Files:',
    '  e5large_vectors.npy     — embedding vectors (numpy)',
    '  chunk_map.pkl           — chunk_ids + id_to_llm + metadata',
    '  bm25_corpus.pkl         — tokenized corpus for BM25',
    '  generation_results.json — all 102 answers + metrics',
    '',
    'Usage (load without recomputing):',
    '  vectors = np.load("e5large_vectors.npy")',
    '  chunk_map = pickle.load(open("chunk_map.pkl", "rb"))',
    '  bm25_tokens = pickle.load(open("bm25_corpus.pkl", "rb"))',
]
summary_path = os.path.join(OUTPUT_DIR, 'artifacts_readme.txt')
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_lines))
print(f'📄 {summary_path}')

# === רשימה + הורדה ===
print(f'\n{"="*60}')
print(f'  📂 {os.path.abspath(OUTPUT_DIR)}/')
print(f'{"="*60}')
total_size = 0
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    fsize = os.path.getsize(fpath)
    total_size += fsize
    print(f'  📄 {fname:<30} {fsize:>12,} bytes')
print(f'  {"":<30} {"─"*12}')
print(f'  {"Total":<30} {total_size:>12,} bytes')

if IN_COLAB:
    print('\n⬇️  מוריד...')
    from google.colab import files as colab_files
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        colab_files.download(os.path.join(OUTPUT_DIR, fname))
    print('✅ כל הקבצים הורדו!')
else:
    print(f'\n✅ נשמר ב: {os.path.abspath(OUTPUT_DIR)}/')


📄 /content/rag_artifacts/e5large_vectors.npy ((1355, 1024), 5,550,208 bytes)
📄 /content/rag_artifacts/chunk_map.pkl (3,384,738 bytes)
📄 /content/rag_artifacts/bm25_corpus.pkl (2,330,470 bytes)
📄 /content/rag_artifacts/generation_results.json (140,576 bytes)
📄 /content/rag_artifacts/artifacts_readme.txt

  📂 /content/rag_artifacts/
  📄 artifacts_readme.txt                    795 bytes
  📄 bm25_corpus.pkl                   2,330,470 bytes
  📄 chunk_map.pkl                     3,384,738 bytes
  📄 e5large_vectors.npy               5,550,208 bytes
  📄 generation_results.json             140,576 bytes
                                 ────────────
  Total                            11,406,787 bytes

⬇️  מוריד...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ כל הקבצים הורדו!


שימוש בהרצה הבאה (ללא חישוב מחדש):

vectors = np.load("e5large_vectors.npy")

chunk_map = pickle.load(open("chunk_map.pkl", "rb"))

bm25_tokens = pickle.load(open("bm25_corpus.pkl", "rb"))
